# COMMUN

### Imports et configuration

In [74]:
import os
import yaml
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine
from dotenv import load_dotenv
from pathlib import Path

### Charger variables d'environnement depuis .env

In [8]:
load_dotenv()

True

In [9]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
print(CONFIG_PATH)

/Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/config.yml


### Définir le chemin racine du projet (quel que soit le dossier courant)

In [10]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

### Lecture du fichier config.yml

In [11]:
test = load_config(CONFIG_PATH)
print(test)

AttributeError: 'NoneType' object has no attribute 'items'

# AHMED

In [12]:
csv = ROOT_DIR / "data" / "acc_2017.csv"
print(csv)

/Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/acc_2017.csv


# ROMAIN

In [13]:
API_CONFIG = {
    'base_url' : 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/accidents-corporels-de-la-circulation-millesime/records',
    'limit_per_request' : 100,
    'max_records' : 1000,
    'timeout' : 30
}

print(f"API: {API_CONFIG['base_url'][:70]}...")

API: https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/acci...


In [25]:
def extraire_accidents_api(max_records=None):
    """
    Fonction pour extraire les accidents depuis l'API
    
    Paramètres:
        max_records (int): Le nombre maximum d'accidents à extraire. Si None, tous les accidents seront extraits.
    
    Return:
        list: Une liste de dictionnaires représentant les accidents extraîts.
    """

print("=" * 80)
print("EXTRACTION DES DONNÉES")
print("=" * 80)

all_records = []
offset = 0
limit = API_CONFIG['limit_per_request']

try:
    #première requête pour connaitre le total
    print("Récupération du nombre total...")
    response = requests.get(
        API_CONFIG['base_url'],
        params={'limit': 1},
        timeout=API_CONFIG['timeout']
    )
    response.raise_for_status()
    data = response.json()
    total_count = data.get('total_count', 0)
    
    print(f"Total disponible: {total_count:,} enregistrements")
    print(data)       
except requests.RequestException as e:
    print(f"\n✗ Erreur lors de l'extraction: {e}")
    raise

EXTRACTION DES DONNÉES
Récupération du nombre total...
Total disponible: 475,911 enregistrements
{'total_count': 475911, 'results': [{'num_acc': '201700009715', 'datetime': '2017-05-28T16:50:00+00:00', 'nom_com': None, 'an': '2017', 'mois': '05', 'jour': '28', 'hrmn': '18:50', 'lum': 'Plein jour', 'agg': 'En agglomération', 'int': '3', 'atm': 'Normale', 'col': 'Deux véhicules – par le coté', 'dep': '13', 'com': '055', 'insee': '13055', 'adr': '6 Av Alexandre  Ansaldi', 'lat': '4333582', 'long': '0539866', 'code_postal': None, 'num': '6', 'coordonnees': {'lon': 2.911777, 'lat': 42.686216}, 'pr': None, 'surf': 'normale', 'v1': None, 'circ': 'Bidirectionnelle', 'vosp': None, 'env1': '00', 'voie': '4', 'larrout': 120, 'v2': None, 'lartpc': 25, 'nbv': 4, 'catr': 'Route Départementale', 'pr1': None, 'plan': 'Partie rectiligne', 'prof': 'Plat', 'infra': None, 'situ': 'Sur chaussée', 'an_nais': ['1998', '1966'], 'sexe': ['Masculin', 'Masculin'], 'actp': ['Se déplaçant', 'Se déplaçant'], 'grav'

In [14]:
"""Téléchargement minimaliste du dataset accidents corporels depuis OpenDataSoft.

Version simplifiée sans retry, sans barre de progression, sans validation.
Télécharge le CSV par chunks et le sauvegarde dans data/accidents_corporels_millesime.csv

Usage:
    python scripts/sauvegarde_csv_api_v1.py

Source:
    https://public.opendatasoft.com - Dataset accidents corporels de la circulation
"""

import sys
from dataclasses import dataclass
from pathlib import Path
import requests



@dataclass(frozen=True)
class Config:
    """Configuration du téléchargement."""
    
    base_url: str = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets"
    dataset_id: str = "accidents-corporels-de-la-circulation-millesime"
    output_dir: str = "data"
    output_file: str = "accidents_corporels_millesime.csv"
    delimiter: str = ","
    chunk_size: int = 65536 #on lit 64KB par 64KB pour ne pas que Lounes voit la RAM de son pc bruler
    timeout: int = 30


def build_url(config: Config) -> str:
    """Construit l'URL de téléchargement."""
    return f"{config.base_url}/{config.dataset_id}/exports/csv?delimiter={config.delimiter}"


def resolve_path(config: Config) -> Path:
    """Détermine le chemin de sortie."""
    #script_dir = ROOT_DIR
    return ROOT_DIR / config.output_dir / config.output_file


def download_csv(url: str, destination: Path, config: Config) -> None:
    """Télécharge le CSV par chunks."""
    print(f"Téléchargement depuis OpenDataSoft...")

    response = requests.get(url, stream=True, timeout=config.timeout)
    response.raise_for_status()

    destination.parent.mkdir(parents=True, exist_ok=True)

    with response, open(destination, "wb") as handle:
        for chunk in response.iter_content(chunk_size=config.chunk_size):
            if chunk:
                handle.write(chunk)

    print(f"Fichier sauvegardé: {destination}")


def main() -> None:
    """Point d'entrée principal."""
    config = Config()
    url = build_url(config)
    path = resolve_path(config)

    download_csv(url, path, config)

    print("Téléchargement terminé")


if __name__ == "__main__":
    main()


Téléchargement depuis OpenDataSoft...


Fichier sauvegardé: /Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
Téléchargement terminé


In [8]:
# Import du script avec chemin absolu
import sys

# Chemin absolu vers le dossier scripts
scripts_dir = ROOT_DIR / 'etl'

# Ajouter au path Python
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

# Import
from sauvegarde_csv_api import download_accidents

# Téléchargement
stats = download_accidents(
    output=ROOT_DIR / 'data/accidents_corporels_millesime.csv',
    delimiter=';',
    show_progress=True,
    retries=3,
    limit = 1000,
    where ="an=2017 AND dep='60'"
)

print(f"✅ Téléchargement réussi !")
print(f"   - {stats.record_count:,} accidents")
print(f"   - {stats.column_count} colonnes")
print(f"   - {stats.size_mb:.2f} MB")


🚗 TÉLÉCHARGEMENT DATASET ACCIDENTS CORPORELS
📍 Source: OpenDataSoft
📅 Période: 2012-2019
📄 Fichier: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
🔤 Séparateur: ';'
🔢 Limite: 1000
🔍 Filtre: an=2017 AND dep='60'

⚠️  Module 'tqdm' non installé - pas de barre de progression détaillée
   Installez-le avec: pip install tqdm

🌐 Connexion à OpenDataSoft...
📥 Téléchargement (taille inconnue)
   Téléchargé: 325.3 KB
✅ Fichier sauvegardé: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
⏱️  Temps de téléchargement: 0.7 secondes

📊 RÉSUMÉ DU TÉLÉCHARGEMENT
✅ Fichier valide
   Taille: 0.32 MB (333,119 bytes)
   Lignes: 435 (incluant l'en-tête)
   Records: 434 accidents
   Colonnes: 69

📋 Premières colonnes:
   1. num_acc
   2. datetime
   3. nom_com
   4. an
   5. mois
   6. jour
   7. hrmn
   8. lum
   9. agg
   10. int
   ... et 59 autres colonnes

✅ Téléc

# LOUNES

In [32]:
# Importation des données source dans un DataFrame pandas

df_source = pd.read_csv(ROOT_DIR / 'data/accidents_corporels_millesime.csv', delimiter=',', low_memory=False)
df_source.head()

,num_acc,datetime,nom_com,an,mois,jour,hrmn,lum,agg,int,...,year_georef,com_name,dep_code,dep_name,epci_code,epci_name,reg_code,reg_name,com_arm_name,com_code
0,201900020750,2019-01-29T15:45:00+00:00,Corbeil-essonnes,2019,1,29,16:45,Plein jour,En agglomération,2,...,2019,Corbeil-Essonnes,91.0,Essonne,200059228.0,CA Grand Paris Sud Seine Essonne Sénart,11.0,Île-de-France,Corbeil-Essonnes,91174.0
1,201900020796,2019-10-07T17:30:00+00:00,Istres,2019,10,7,19:30,Nuit sans éclairage public,Hors agglomération,1,...,2019,Istres,13.0,Bouches-du-Rhône,200054807.0,Métropole d'Aix-Marseille-Provence,93.0,Provence-Alpes-Côte d'Azur,Istres,13047.0
2,201900020869,2019-10-13T13:46:00+00:00,Saint-laurent-du-pont,2019,10,13,15:46,Plein jour,En agglomération,1,...,2019,Saint-Laurent-du-Pont,38.0,Isère,200040111.0,CC Coeur de Chartreuse,84.0,Auvergne-Rhône-Alpes,Saint-Laurent-du-Pont,38412.0
3,201900021309,2019-04-17T14:30:00+00:00,Livry-gargan,2019,4,17,16:30,Plein jour,En agglomération,1,...,2019,Livry-Gargan,93.0,Seine-Saint-Denis,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Livry-Gargan,93046.0
4,201900018753,2019-12-25T17:10:00+00:00,Gennevilliers,2019,12,25,18:10,Nuit sans éclairage public,Hors agglomération,1,...,2019,Gennevilliers,92.0,Hauts-de-Seine,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Gennevilliers,92036.0


In [33]:
# Exploration de la donnée source csv

df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 69 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   num_acc       475911 non-null  int64  
 1   datetime      475911 non-null  object 
 2   nom_com       449206 non-null  object 
 3   an            475911 non-null  int64  
 4   mois          475911 non-null  int64  
 5   jour          475911 non-null  int64  
 6   hrmn          475911 non-null  object 
 7   lum           475911 non-null  object 
 8   agg           475911 non-null  object 
 9   int           475911 non-null  int64  
 10  atm           475861 non-null  object 
 11  col           475901 non-null  object 
 12  dep           475911 non-null  object 
 13  com           475911 non-null  object 
 14  insee         475296 non-null  float64
 15  adr           426655 non-null  object 
 16  lat           300785 non-null  object 
 17  long          300785 non-null  object 
 18  code

In [ ]:
"""
Préparation des données : nettoyage, transformation

Objectif : diviser le csv en 5 tables distinctes (accident, vehicule, usager, lieux, date)
Chaque table sera dans une dataframe pandas distincte

1ere étape : premier petit nettoyage et uniformisation des données
2eme étape : séparation en 5 dataframes
3ème étape : explosion des colonnes multi-valeurs pour les dataframes vehicule et usager
4eme étape : nettoyage spécifique à chaque dataframe
5ème étape : mapping des valeurs catégorielles pour chaque dataframe
6ème étape : tests de validation des données
7ème étape : export des dataframes nettoyés et validés dans postgres
"""

# Premier nettoyage et uniformisation des données
df_source.columns = df_source.columns.str.lower().str.strip()
df_source["datetime"] = pd.to_datetime(df_source["datetime"], errors="coerce")

array(['Normale', 'Temps couvert', 'Temps éblouissant', 'Pluie légère',
       'Pluie forte', 'Autre', 'Vent fort - tempête',
       'Brouillard - fumée', 'Neige - grêle', nan, '-1'], dtype=object)

In [78]:
# Séparation en 5 dataframes
# 1. Accidents
df_accidents = df_source[[
    'num_acc', 
    'lum', 
    'agg',
    'int',
    'atm',
    'adr',
    'col',
    'circ',
    'plan',
    'prof',
    'surf',
    'infra',
    'situ',
    'year_georef'
]].copy()

# df_accidents.head()
df_accidents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   num_acc      475911 non-null  int64 
 1   lum          475911 non-null  object
 2   agg          475911 non-null  object
 3   int          475911 non-null  int64 
 4   atm          475861 non-null  object
 5   adr          426655 non-null  object
 6   col          475901 non-null  object
 7   circ         450504 non-null  object
 8   plan         441487 non-null  object
 9   prof         447014 non-null  object
 10  surf         459780 non-null  object
 11  infra        54500 non-null   object
 12  situ         449540 non-null  object
 13  year_georef  475911 non-null  int64 
dtypes: int64(3), object(11)
memory usage: 50.8+ MB


In [79]:
# Tests de validation dataframe df_accidents

# Vérification des doublons sur 'num_acc
duplicates_accidents = df_accidents.duplicated(subset=['num_acc']).sum()
print(f"Doublons dans df_accidents sur 'num_acc': {duplicates_accidents} \n")

# Vérification des valeurs uniques sur les catégories
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    unique_values = df_accidents[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    nan_count = df_accidents[col].isna().sum()
    neg_one_count = (df_accidents[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


Doublons dans df_accidents sur 'num_acc': 0 


 Valeurs uniques dans 'lum': ['Plein jour' 'Nuit sans éclairage public'
 'Nuit avec éclairage public non allumé'
 'Nuit avec éclairage public allumé' 'Crépuscule ou aube']

 Valeurs uniques dans 'agg': ['En agglomération' 'Hors agglomération']

 Valeurs uniques dans 'int': [2 1 9 6 3 7 4 5 0 8]

 Valeurs uniques dans 'atm': ['Normale' 'Temps couvert' 'Temps éblouissant' 'Pluie légère'
 'Pluie forte' 'Autre' 'Vent fort - tempête' 'Brouillard - fumée'
 'Neige - grêle' nan '-1']

 Valeurs uniques dans 'col': ['Deux véhicules – par le coté' 'Sans collision'
 'Deux véhicules - frontale' 'Deux véhicules – par l’arrière'
 'Autre collision' 'Trois véhicules et plus - collisions multiples'
 'Trois véhicules et plus – en chaîne' nan '-1']

 Valeurs uniques dans 'circ': ['Bidirectionnelle' '-1' 'A sens unique' 'A chaussées séparées' nan
 'Avec voies d’affectation variable']

 Valeurs uniques dans 'plan': ['Partie rectiligne' nan 'En courbe à droite' 

In [80]:
# Transformation des valeurs '-1' en NaN pour les colonnes catégorielles
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    df_accidents[col] = df_accidents[col].replace('-1', np.nan)

In [51]:
# Séparation en 5 dataframes
# 2. Lieux
df_lieux = df_source[[
    'com_code',
    'com_name',
    'dep_code',
    'dep_name',
    'reg_code', 
    'reg_name', 
    'epci_code', 
    'epci_name',
    'lat', 
    'long', 
    'catr', 
    'v1', 
    'voie', 
    'v2', 
    'nbv', 
    'vosp', 
    'pr', 
    'pr1', 
    'lartpc', 
    'larrout', 
    'num_acc'
]].copy()

# df_lieux.head()
df_lieux.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 21 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   com_code   475296 non-null  float64
 1   com_name   464648 non-null  object 
 2   dep_code   464648 non-null  float64
 3   dep_name   464648 non-null  object 
 4   reg_code   464648 non-null  float64
 5   reg_name   464648 non-null  object 
 6   epci_code  426190 non-null  float64
 7   epci_name  426190 non-null  object 
 8   lat        300785 non-null  object 
 9   long       300785 non-null  object 
 10  catr       475911 non-null  object 
 11  v1         50267 non-null   float64
 12  voie       410712 non-null  object 
 13  v2         19498 non-null   object 
 14  nbv        474103 non-null  float64
 15  vosp       32080 non-null   object 
 16  pr         259060 non-null  object 
 17  pr1        250140 non-null  float64
 18  lartpc     362782 non-null  float64
 19  larrout    364392 non-n

In [50]:
# Séparation en 5 dataframes
# 3. Date_accident
df_date_accident = df_source[[
    'datetime', 
    'an', 
    'mois', 
    'jour', 
    'hrmn', 
    'num_acc'
]].copy()

# df_date_accident.head()
df_date_accident.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   datetime  475911 non-null  datetime64[ns, UTC]
 1   an        475911 non-null  int64              
 2   mois      475911 non-null  int64              
 3   jour      475911 non-null  int64              
 4   hrmn      475911 non-null  object             
 5   num_acc   475911 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 21.8+ MB


In [ ]:
# Séparation en 5 dataframes
# 4. Vehicules
df_vehicules = df_source[[
    'num_veh', 
    'catv', 
    'choc', 
    'senc', 
    'obs', 
    'obsm', 
    'occutc', 
    'manv', 
    'num_acc'
]].copy()

df_vehicules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 9 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   num_veh  475911 non-null  object
 1   catv     475911 non-null  object
 2   choc     446407 non-null  object
 3   senc     209666 non-null  object
 4   obs      101364 non-null  object
 5   obsm     369261 non-null  object
 6   occutc   6015 non-null    object
 7   manv     441491 non-null  object
 8   num_acc  475911 non-null  int64 
dtypes: int64(1), object(8)
memory usage: 32.7+ MB


In [54]:
# Séparation en 5 dataframes
# 5. Usagers
df_usagers = df_source[[
    'sexe', 
    'grav', 
    'trajet', 
    'secu', 
    'secu_utl', 
    'catu', 
    'place', 
    'locp', 
    'actp', 
    'etatp', 
    'num_acc'
]].copy()

df_usagers.head()
#df_usagers.info()

,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc
0,"Masculin,Masculin,Masculin","Blessé,Blessé,Indemne","Promenade – loisirs,Promenade – loisirs",NaN,NaN,"Conducteur,Passager,Conducteur","1,2,1",-1,"-1,Se déplaçant,Se déplaçant","-1,-1,-1",201900020750
1,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,-1,-1,-1,201900020796
2,"Masculin,Masculin,Masculin,Masculin,Masculin","Indemne,Indemne,Blessé,Indemne,Indemne","Promenade – loisirs,Promenade – loisirs,Promen...",NaN,NaN,"Conducteur,Passager,Conducteur,Passager,Passager","1,9,1,7,2",NaN,"Se déplaçant,Se déplaçant,Se déplaçant,Se dépl...","-1,-1,-1,-1,-1",201900020869
3,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,-1,-1,-1,201900021309
4,"Féminin,Masculin,Masculin,Féminin,Masculin,Fém...","Indemne,Blessé,Indemne,Indemne,Indemne,Indemne","Promenade – loisirs,Promenade – loisirs,Promen...",NaN,NaN,"Conducteur,Passager,Passager,Passager,Passager...","1,2,2,3,4,1","-1,-1,-1,-1,-1,-1","Se déplaçant,Se déplaçant,Se déplaçant,Se dépl...","-1,-1,-1,-1,-1,-1",201900018753


# ZOUBIR